# Grid Search 與分類閾值調整

1. 讀取中風資料集並切分訓練集、測試集。
2. 建立前處理與 Logistic Regression Pipeline。
3. 使用 `GridSearchCV` 尋找超參數。
4. 只使用訓練資料的 OOF 預測機率尋找最佳 F2 threshold。
5. 在完全未參與調參的測試集比較 threshold `0.5` 與調整後 threshold。

> 重點：**模型超參數**與**分類 threshold**是兩件不同的事。Grid Search 選模型設定；threshold tuning 決定機率多高才判為中風。


## 1. 載入套件與設定

`RANDOM_STATE` 固定隨機結果，讓每次執行得到可重現的資料切分與交叉驗證結果。


In [21]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = Path("healthcare-dataset-stroke-data.csv")

## 2. 讀取資料

- `id` 只是識別碼，不作為模型特徵。
- `stroke` 是要預測的目標欄位。
- `stratify=y` 讓訓練集和測試集維持相近的中風比例，這對不平衡資料很重要。


In [22]:
df = pd.read_csv(DATA_PATH)

X = df.drop(columns=["id", "stroke"])
y = df["stroke"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Full dataset rows: {len(df)}")
print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Full stroke rate: {y.mean():.4f}")
print(f"Training stroke rate: {y_train.mean():.4f}")
print(f"Test stroke rate: {y_test.mean():.4f}")

df.head()

Full dataset rows: 5110
Training rows: 4088
Test rows: 1022
Full stroke rate: 0.0487
Training stroke rate: 0.0487
Test stroke rate: 0.0489


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. 區分數值與類別特徵

不同資料型態需要不同前處理：

- 數值特徵：以中位數補缺值，再標準化。
- 類別特徵：以眾數補缺值，再做 One-Hot Encoding。


In [23]:
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Numeric features: ", numeric_features)
print("Categorical features: ", categorical_features)

Numeric features:  ['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi']
Categorical features:  ['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status']


## 4. 建立前處理與模型 Pipeline

Pipeline 的優點是交叉驗證的每一折都會獨立進行補值、標準化與編碼，可以避免把驗證資料的資訊洩漏到訓練過程。


In [24]:
def build_pipeline(numeric_features, categorical_features):
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
            ),
        ]
    )


pipeline = build_pipeline(numeric_features, categorical_features)
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

## 5. 使用 GridSearchCV 尋找模型超參數

搜尋的參數：

- `C`：正則化強度的反比；越小代表正則化越強。
- `class_weight`：提高少數類別 `stroke=1` 的重要性。
- `solver`：Logistic Regression 的最佳化演算法。

評分使用 `average_precision`，也就是 PR-AUC。中風樣本很少時，PR-AUC 通常比 Accuracy 更能反映模型辨識少數類別的能力。


In [25]:
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0, 100.0],
    "model__class_weight": [
        None,
        "balanced",
        {0: 1, 1: 3},
        {0: 1, 1: 5},
        {0: 1, 1: 10},
    ],
    "model__solver": ["liblinear", "lbfgs"],
}

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="average_precision",
    cv=inner_cv,
    n_jobs=1,
    refit=True,
    return_train_score=False,
)

search.fit(X_train, y_train)

print("Best parameters: ")
print(search.best_params_)
print(f"Best cross-validation PR-AUC: {search.best_score_:.4f}")

Best parameters: 
{'model__C': 0.01, 'model__class_weight': {0: 1, 1: 5}, 'model__solver': 'liblinear'}
Best cross-validation PR-AUC: 0.1958


### 查看表現最好的參數組合

這裡把 Grid Search 結果依照驗證集 PR-AUC 排序，方便觀察不同參數的差異。


### 圖表：GridSearchCV 參數搜尋結果

![GridSearchCV 各參數組合的交叉驗證 PR-AUC](extension_report_assets/01_gridsearch_pr_auc_heatmap.png)

**圖表解讀：**

- 顏色越深代表五折交叉驗證的 PR-AUC 越高。
- 最佳參數為 `C=0.01`、`class_weight={0:1, 1:5}`、`solver='liblinear'`。
- 最佳交叉驗證 PR-AUC 為 `0.1958`。
- 這表示適度提高中風類別權重，比完全不加權更有利於辨識少數類別。

**八分鐘報告講法：**  
本研究不是只測試單一組參數，而是利用 GridSearchCV 比較 50 種組合，並以適合不平衡資料的 PR-AUC 作為選擇標準。


In [26]:
grid_results = (
    pd.DataFrame(search.cv_results_)
    .sort_values("rank_test_score")
    [[
        "rank_test_score",
        "mean_test_score",
        "std_test_score",
        "param_model__C",
        "param_model__class_weight",
        "param_model__solver",
    ]]
)

grid_results.head(10)

,rank_test_score,mean_test_score,std_test_score,param_model__C,param_model__class_weight,param_model__solver
6,1,0.195785,0.049240,0.01,"{0: 1, 1: 5}",liblinear
4,2,0.195266,0.051798,0.01,"{0: 1, 1: 3}",liblinear
47,3,0.194109,0.039479,100.00,"{0: 1, 1: 5}",lbfgs
36,4,0.193993,0.039513,10.00,"{0: 1, 1: 5}",liblinear
49,5,0.193982,0.038853,100.00,"{0: 1, 1: 10}",lbfgs
46,6,0.193976,0.039493,100.00,"{0: 1, 1: 5}",liblinear
37,7,0.193922,0.039163,10.00,"{0: 1, 1: 5}",lbfgs
39,8,0.193893,0.038730,10.00,"{0: 1, 1: 10}",lbfgs
38,9,0.193892,0.038763,10.00,"{0: 1, 1: 10}",liblinear
48,10,0.193781,0.038742,100.00,"{0: 1, 1: 10}",liblinear


## 6. 用 OOF 預測機率調整 threshold

預設情況下，機率大於等於 `0.5` 才預測為中風。但在醫療篩檢中，漏掉真正中風者的代價通常較高，因此可降低 threshold 來提高 Recall。

不能直接用測試集找 threshold，否則測試集就不再是公平的最終評估。這裡使用訓練集的 Out-of-Fold（OOF）預測：每筆訓練資料的機率，都是由「沒有看過該筆資料」的模型產生。


In [27]:
threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE + 1,
)

oof_probabilities = cross_val_predict(
    clone(search.best_estimator_),
    X_train,
    y_train,
    cv=threshold_cv,
    method="predict_proba",
    n_jobs=1,
)[:, 1]

print("First 10 OOF probabilities:")
print(np.round(oof_probabilities[:10], 4))

First 10 OOF probabilities:
[0.1178 0.0424 0.2688 0.1285 0.0414 0.4693 0.5413 0.085  0.0379 0.3064]


## 7. 尋找 F2-score 最高的 threshold

F2-score 比 F1-score 更重視 Recall：

\[
F_2 = \frac{5 \times Precision \times Recall}{4 \times Precision + Recall}
\]

因此它適合「寧可多找出一些疑似個案，也不要漏掉真正陽性」的情境。


### 圖表：Precision、Recall 與 F2-score 的門檻變化

![不同分類 threshold 下的 Precision、Recall 與 F2-score](extension_report_assets/02_threshold_precision_recall_f2.png)

**圖表解讀：**

- threshold 越高，模型越不容易判定為中風，因此 Recall 逐漸下降。
- threshold 降低可以找出更多真正中風個案，但 Precision 也可能下降。
- F2-score 比 F1-score 更重視 Recall，最高點對應的 threshold 約為 `0.268`。
- 門檻只使用訓練集 OOF 預測決定，測試集沒有參與選擇，可避免資料洩漏。

**八分鐘報告講法：**  
預設門檻 0.5 不一定適合醫療篩檢，因此本研究使用 F2-score，在重視 Recall 的前提下選出約 0.268 的門檻。


In [28]:
def find_best_f2_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(
        y_true, probabilities
    )

    # thresholds has one fewer item, so use [:-1] to align the arrays.
    f2_scores = (
        5 * precision[:-1] * recall[:-1]
        / np.maximum(4 * precision[:-1] + recall[:-1], 1e-12)
    )

    best_index = int(np.argmax(f2_scores))
    return {
        "threshold": float(thresholds[best_index]),
        "precision": float(precision[best_index]),
        "recall": float(recall[best_index]),
        "f2": float(f2_scores[best_index]),
    }


threshold_result = find_best_f2_threshold(y_train, oof_probabilities)
pd.Series(threshold_result).round(4)

threshold    0.2680
precision    0.1453
recall       0.7688
f2           0.4137
dtype: float64

## 8. 在未碰過的測試集進行最終評估

先重新用完整訓練集訓練最佳模型，再比較：

- 預設 threshold：`0.5`
- 調整後 threshold：由訓練資料的 OOF 預測決定

PR-AUC 與 ROC-AUC 只依賴預測機率，因此兩列會相同；Accuracy、Precision、Recall、F1、F2 則會隨 threshold 改變。


In [29]:
def metric_row(y_true, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)

    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(y_true, predictions),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "f2": fbeta_score(
            y_true, predictions, beta=2, zero_division=0
        ),
        "pr_auc": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }, predictions


final_model = clone(search.best_estimator_)
final_model.fit(X_train, y_train)
test_probabilities = final_model.predict_proba(X_test)[:, 1]

default_metrics, default_predictions = metric_row(
    y_test,
    test_probabilities,
    threshold=0.5,
)
tuned_metrics, tuned_predictions = metric_row(
    y_test,
    test_probabilities,
    threshold=threshold_result["threshold"],
)

comparison = pd.DataFrame(
    [default_metrics, tuned_metrics],
    index=["Default threshold", "Tuned threshold"],
)
comparison.round(4)

,threshold,accuracy,precision,recall,f1,f2,pr_auc,roc_auc
Default threshold,0.500,0.9198,0.2647,0.36,0.3051,0.3358,0.2517,0.8387
Tuned threshold,0.268,0.7857,0.1606,0.80,0.2676,0.4454,0.2517,0.8387


## 9. 混淆矩陣與分類報告

混淆矩陣格式為：

```text
[[TN, FP],
 [FN, TP]]
```

調低 threshold 後，通常會得到更多 TP、較少 FN，但也可能增加 FP。這正是 Recall 與 Precision 之間的取捨。


### 圖表：分類門檻調整前後的混淆矩陣

![預設與調整後 threshold 的混淆矩陣比較](extension_report_assets/03_confusion_matrix_comparison.png)

**圖表解讀：**

- True Positive（成功找出的中風個案）由 `18` 人增加到 `40` 人。
- False Negative（漏判）由 `32` 人下降到 `10` 人。
- False Positive（誤警示）由 `50` 人增加到 `209` 人。
- 模型降低了漏判風險，但會讓更多未中風者被列為高風險。

**八分鐘報告講法：**  
調整門檻後多找出 22 位真正中風者，並將漏判減少 22 人；代價是誤警示數量明顯增加。


In [30]:
print("Default threshold confusion matrix:")
print(confusion_matrix(y_test, default_predictions))

print("\nTuned threshold confusion matrix:")
print(confusion_matrix(y_test, tuned_predictions))

print("\nTuned threshold classification report:")
print(
    classification_report(
        y_test,
        tuned_predictions,
        digits=4,
        zero_division=0,
    )
)

Default threshold confusion matrix:
[[922  50]
 [ 32  18]]

Tuned threshold confusion matrix:
[[763 209]
 [ 10  40]]

Tuned threshold classification report:
              precision    recall  f1-score   support

           0     0.9871    0.7850    0.8745       972
           1     0.1606    0.8000    0.2676        50

    accuracy                         0.7857      1022
   macro avg     0.5739    0.7925    0.5710      1022
weighted avg     0.9466    0.7857    0.8448      1022



## 10. 最終結果與醫療篩檢取捨

![Threshold tuning 前後的模型指標與誤判比較](extension_report_assets/04_performance_tradeoff.png)

**主要結果：**

| 指標 | 預設 threshold 0.5 | 調整後 threshold 0.268 |
|---|---:|---:|
| Accuracy | 0.92 | 0.79 |
| Precision | 0.26 | 0.16 |
| Recall | 0.36 | 0.80 |
| F1-score | 0.31 | 0.27 |
| F2-score | 0.34 | 0.45 |
| False Negative | 32 | 10 |
| False Positive | 50 | 209 |

**研究結論：**

分類門檻調整能明顯提高中風個案的檢出能力。若研究目的為早期風險篩檢，Recall 從 `0.36` 提升至 `0.80`、FN 從 `32` 降至 `10`，具有實際意義；但 Precision 與 Accuracy 下降，且誤警示增加，因此模型應定位為輔助篩檢工具，而不是臨床診斷工具。


## 結論閱讀方式

執行完後，建議依序回答下列問題：

1. Grid Search 選到了哪一組 `C`、`class_weight`、`solver`？
2. OOF 方法找到的最佳 threshold 是否低於 `0.5`？
3. 調整 threshold 後 Recall 提高多少？
4. Precision 和 Accuracy 是否因此下降？
5. 在此中風預測情境中，提高 Recall 所付出的代價是否可以接受？

這份 Notebook 與原始 `.py` 使用相同的核心流程，但將每個階段拆開，方便觀察變數與理解模型決策。
